In [4]:
import pandas as pd
import numpy as np
# Lire le fichier CSV
df = pd.read_csv('data/data_CoM_CenA_M83_0.64_lg.csv')

# Filtrer pour ne garder que les lignes où ref_V_h vaut "LEDA"
# df_filtre = df[df['ref_V_h'] == 'LEDA']



df_filtre = df[
    # (df['ref_V_h'] == 'LEDA') &
    (df['dis_center_CoM_CenA_M83_0.64'] < 6) 
    # (df['minor_infall_CoM_CenA_M83_0.77'] > -45*np.pi/180)
    # (df['Dec']< 0)
]

   #  df_filtre = df[
   #      (df["dis_center_CoM_CenA_M83_0.76"] < 7 )
   #  ]


# Sauvegarder le résultat
df_filtre.to_csv('data/data_CoM_CenA_M83_0.64_lg_filtre_distance.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
# Lire le fichier CSV
df = pd.read_csv('data/data_CoM_CenA_M83_0.77_lg.csv')

# Filtrer pour ne garder que les lignes où ref_V_h vaut "LEDA"
# df_filtre = df[df['ref_V_h'] == 'LEDA']



df_filtre = df[
    (df['ref_V_h'] == 'LEDA') &
    (df['angular_distance_CoM_CenA_M83_0.77'] < 45*np.pi/180) &
    (df['angular_distance_CoM_CenA_M83_0.77'] > -45*np.pi/180)
    # (df['Dec']< 0)
]

   #  df_filtre = df[
   #      (df["dis_center_CoM_CenA_M83_0.76"] < 7 )
   #  ]


# Sauvegarder le résultat
df_filtre.to_csv('data/data_CoM_CenA_M83_0.77_lg_filtre_angular.csv', index=False)

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/new_data_lg.csv')
df_filtre = df[
    (df['ref_V_h'] == 'LEDA')
]


# Sauvegarder le résultat
df_filtre.to_csv('data/new_data_lg_without_muse.csv', index=False)

In [1]:
import numpy as np
import pandas as pd

def spherical_to_cartesian(ra_deg, dec_deg, r):
    ra_rad = np.radians(ra_deg)
    dec_rad = np.radians(dec_deg)
    x = r * np.cos(dec_rad) * np.cos(ra_rad)
    y = r * np.cos(dec_rad) * np.sin(ra_rad)
    z = r * np.sin(dec_rad)
    return np.stack((x, y, z), axis=-1)

def add_column_lambda(df, CoM_name):
    """
    Ajoute une colonne 'lambda' (λ), l'angle entre la ligne de visée vers une galaxie
    et la direction du centre du cluster (depuis l'observateur).
    """
    galaxy_pos = spherical_to_cartesian(df['RA'], df['Dec'], df['Dis'])

    center_row = df[df["Name"] == CoM_name]
    if center_row.empty:
        raise ValueError(f"Cluster center '{CoM_name}' not found in DataFrame.")
    center_pos = spherical_to_cartesian(
        center_row["RA_radians"].values[0],
        center_row["Dec_radians"].values[0],
        center_row["Dis"].values[0]
    )

    # Vecteurs depuis l'observateur vers la galaxie et vers le centre
    vec_GO = galaxy_pos
    vec_CO = np.tile(center_pos, (len(df), 1))

    # Calcul angle entre les directions
    dot_products = np.einsum('ij,ij->i', vec_GO, vec_CO)
    norm_GO = np.linalg.norm(vec_GO, axis=1)
    norm_CO = np.linalg.norm(vec_CO, axis=1)
    cos_lambda = dot_products / (norm_GO * norm_CO)
    cos_lambda = np.clip(cos_lambda, -1.0, 1.0)
    lambda_rad = np.arccos(cos_lambda)
    lambda_deg = 180.0 - np.degrees(lambda_rad)  # Pour correspondre à la définition

    df['lambda'] = lambda_deg
    return df

CoM_Name="CoM_CenA_M83_0.77" 
galaxy_df = pd.read_csv('data/data_CoM_CenA_M83_0.77_lg.csv')    
add_column_lambda(galaxy_df,CoM_name=CoM_Name)
# galaxy_df.to_csv('data/data_CoM_CenA_M83_0.77_lg_filtre_lambda.csv', index=False)


,Unnamed: 0,Name,PGC,RA,Dec,Dis,e_Dis_min,e_Dis_max,ref_dis,V_h,...,dis_center_CoM_CenA_M83_0.77,e_dis_center_min_CoM_CenA_M83_0.77,e_dis_center_max_CoM_CenA_M83_0.77,major_infall_velocity_CoM_CenA_M83_0.77,e_major_infall_velocity_min_CoM_CenA_M83_0.77,e_major_infall_velocity_max_CoM_CenA_M83_0.77,minor_infall_velocity_CoM_CenA_M83_0.77,e_minor_infall_velocity_min_CoM_CenA_M83_0.77,e_minor_infall_velocity_max_CoM_CenA_M83_0.77,lambda
0,0,PGC1599237,PGC1599237,138.413100,19.618800,8.128305,0.330004,0.343969,CF4,322.748756,...,7.297393,0.292432,0.305852,4209.962535,2021.899450,18315.306491,295.483183,6.580086,6.816478,48.334506
1,1,PGC166152,PGC166152,196.259700,-40.082300,5.807644,0.337485,0.358306,CF4,361.935503,...,1.998790,0.396840,0.420248,59.227732,7.991699,8.704211,65.111183,7.640164,7.627211,41.739781
2,2,KK189,PGC166158,198.188300,-41.832000,4.210000,0.170000,0.170000,-6,501.463158,...,0.441609,0.146314,0.205163,263.412313,50.492408,425.006621,170.200911,80.983952,27.855456,43.886952
3,3,PGC166163,PGC166163,200.284500,-31.529100,5.495409,0.247334,0.258991,CF4,344.571324,...,1.632734,0.317786,0.328595,41.137501,8.179852,8.257293,41.906669,8.152441,8.153806,35.306097
4,4,KK203,PGC166167,201.868100,-45.352400,3.780000,0.250000,0.250000,-3,57.409329,...,0.092683,0.070006,0.319779,253.491021,5333.493102,1311.630343,238.992968,498.770488,21.863534,48.171482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,153,dw1323-40a,NaN,201.223300,-40.761200,3.730000,0.150000,0.150000,-10,207.099928,...,0.153850,0.089714,0.213517,105.637630,1407.284554,5668.759927,89.451426,176.942240,25.025496,43.824243
154,154,dw1323-40b,NaN,200.980900,-40.836100,3.910000,0.610000,0.610000,-10,253.407148,...,0.091241,0.010660,0.634768,-116.675174,2165.853953,2651.064253,-16.735828,49.244404,84.121233,43.811627
155,155,dw1341-43,NaN,205.403200,-43.855300,3.530000,0.040000,0.040000,-10,397.657080,...,0.398205,0.089530,0.099313,-107.805177,40.152371,26.797973,-70.049103,24.365474,27.345679,48.007698
156,156,dw1342-43,NaN,205.683700,-43.254800,2.900000,0.140000,0.140000,-10,273.317743,...,0.991214,0.204151,0.207675,31.193311,13.389267,13.831393,33.174037,13.312542,13.163639,47.592613


In [ ]:
galaxy_df_filtre = galaxy_df[
#(galaxy_df['ref_V_h'] == 'LEDA') & 
    (
        (galaxy_df['lambda'] >= 135) |
        (galaxy_df['lambda'] <= 45) |
        (galaxy_df['dis_center_CoM_CenA_M83_0.77'] < 2)
    )
]
galaxy_df_filtre.to_csv('data/data_CoM_CenA_M83_0.77_lg_filtre_lambda.csv', index=False)

: 

In [ ]:
import numpy as np
import pandas as pd

CoM_Name="CoM_CenA_M83_0.76" 
galaxy_df = pd.read_csv('data/data_CoM_CenA_M83_0.76_lg.csv')    

def velocity_model_1(r, H0, M):
    G = 4.3009e-9  # Constante gravitationnelle en (km/s)² Mpc / M☉
    Omega_Lambda = 0.67

    # Avoid numerical issues with the square root
    if M <= 0:
        return np.full_like(r, np.nan)  # avoid sqrt of negative mass
    
    term1 = H0 * (1.1 + 0.31 * Omega_Lambda) * r
    
    # Check if the term inside sqrt will be negative
    sqrt_term = G * M / r
    valid_points = sqrt_term > 0
    
    result = np.full_like(r, np.nan)
    if np.any(valid_points):
        result[valid_points] = term1[valid_points] - 1.1 * np.sqrt(sqrt_term[valid_points])
        
    return result

import pandas as pd
import numpy as np


def symmetrize_and_adjust_errors(df,row_name):
    """
    Modifie le DataFrame en rendant les erreurs sur la distance symétriques
    avec la plus grande des deux erreurs initiales, et en s'assurant que
    l'erreur finale est au moins 10% de la valeur de distance_from_center.

    Paramètres :
        df (pd.DataFrame) : DataFrame contenant les colonnes suivantes :
            - distance_from_center
            - e_distance_from_center_min
            - e_distance_from_center_max

    Retour :
        pd.DataFrame : le DataFrame modifié (in-place)
    """
    #dist
#     # Calcul de l'erreur maximale entre les deux erreurs
#     max_err_dis = df[['e_dis_center_min_' + row_name, 'e_dis_center_max_' + row_name]].max(axis=1)
# 
#     # Calcul de l'erreur minimale autorisée (10% de la distance)
#     min_allowed_err_dis = np.maximum(df['dis_center_' + row_name].abs() * 0.10,1)
# 
#     # Erreur finale : le max entre l'erreur initiale et 10% de la distance
#     final_err_dis = np.maximum(max_err_dis, min_allowed_err_dis)
# 
#
#    # minor
#    # Remplacement des colonnes d'erreur par l'erreur symétrique ajustée
#    df['e_dis_center_min_' + row_name] = final_err_dis
#    df['e_dis_center_max_' + row_name] = final_err_dis

    max_err_minor = df[['e_minor_infall_velocity_min_' + row_name, 'e_minor_infall_velocity_max_' + row_name]].max(axis=1)

    # Calcul de l'erreur minimale autorisée (10% de la distance)
    min_allowed_err_minor = np.maximum(df['minor_infall_velocity_' + row_name].abs() * 0.10,50)

    # Erreur finale : le max entre l'erreur initiale et 10% de la distance
    final_err_minor = np.maximum(max_err_minor, min_allowed_err_minor)

    # Remplacement des colonnes d'erreur par l'erreur symétrique ajustée
    df['e_minor_infall_velocity_min_'+row_name] = final_err_minor
    df['e_minor_infall_velocity_max_'+row_name] = final_err_minor
    
    
    # MAJOR
    max_err_major = df[['e_major_infall_velocity_min_' + row_name, 'e_major_infall_velocity_max_' + row_name]].max(axis=1)

    # Calcul de l'erreur minimale autorisée (10% de la distance)
    min_allowed_err_major = np.maximum(df['major_infall_velocity_' + row_name].abs() * 0.10,75)

    # Erreur finale : le max entre l'erreur initiale et 10% de la distance
    final_err_major = np.maximum(max_err_major, min_allowed_err_major)

    # Remplacement des colonnes d'erreur par l'erreur symétrique ajustée
    df['e_major_infall_velocity_min_' + row_name] = final_err_major
    df['e_major_infall_velocity_max_' + row_name] = final_err_major

    return df


galaxy_df_big_errors = symmetrize_and_adjust_errors(galaxy_df,row_name=CoM_Name)
galaxy_df_big_errors.to_csv('data/data_CoM_CenA_M83_0.76_lg_big_errors_v.csv', index=False)

In [2]:
import numpy as np
import pandas as pd

CoM_Name="CoM_CenA_M83_0.64" 
galaxy_df = pd.read_csv('data/data_CoM_CenA_M83_0.64_lg.csv')    

def symmetrize_and_adjust_errors(df,row_name):
    """
    Modifie le DataFrame en rendant les erreurs sur la distance symétriques
    avec la plus grande des deux erreurs initiales, et en s'assurant que
    l'erreur finale est au moins 10% de la valeur de distance_from_center.

    Paramètres :
        df (pd.DataFrame) : DataFrame contenant les colonnes suivantes :
            - distance_from_center
            - e_distance_from_center_min
            - e_distance_from_center_max

    Retour :
        pd.DataFrame : le DataFrame modifié (in-place)
    """



    # Remplacement des colonnes d'erreur par l'erreur symétrique ajustée
    df['e_minor_infall_velocity_min_'+row_name] =  df['e_minor_infall_velocity_min_'+row_name] + 60.5
    df['e_minor_infall_velocity_max_'+row_name] =  df['e_minor_infall_velocity_max_'+row_name] + 60.5
    
    

    # Remplacement des colonnes d'erreur par l'erreur symétrique ajustée
    df['e_major_infall_velocity_min_' + row_name] = df['e_major_infall_velocity_min_' + row_name] + 83.5
    df['e_major_infall_velocity_max_' + row_name] = df['e_major_infall_velocity_max_' + row_name] + 83.5

    return df


galaxy_df_big_errors = symmetrize_and_adjust_errors(galaxy_df,row_name=CoM_Name)
galaxy_df_big_errors.to_csv('data/data_CoM_CenA_M83_0.64_lg_big_errors_v_sigma.csv', index=False)

In [6]:
import numpy as np
import pandas as pd

def velocity_model_1(r, H0, M):
    G = 4.3009e-9  # Constante gravitationnelle en (km/s)² Mpc / M☉
    Omega_Lambda = 0.67

    # Avoid numerical issues with the square root
    if M <= 0:
        return np.full_like(r, np.nan)  # avoid sqrt of negative mass
    
    term1 = H0 * (1.1 + 0.31 * Omega_Lambda) * r
    
    # Check if the term inside sqrt will be negative
    sqrt_term = G * M / r
    valid_points = sqrt_term > 0
    
    result = np.full_like(r, np.nan)
    if np.any(valid_points):
        result[valid_points] = term1[valid_points] - 1.1 * np.sqrt(sqrt_term[valid_points])
        
    return result

def add_diff_column(df,row_name):
    df["diff_major"] = np.absolute(df['major_infall_velocity_' + row_name]-velocity_model_1(df['dis_center_' + row_name],70,3*10**12))
    df["diff_minor"] = np.absolute(df['minor_infall_velocity_' + row_name]-velocity_model_1(df['dis_center_' + row_name],65,1*10**12))
    return df

CoM_Name="CoM_CenA_M83_0.77" 
galaxy_df = pd.read_csv('data/data_CoM_CenA_M83_0.77_lg.csv')
galaxy_df_diff= add_diff_column(galaxy_df,CoM_Name)
galaxy_df_diff.to_csv('data/data_CoM_CenA_M83_0.77_lg_diff_vel.csv', index=False)